# Baseline UPD on real data

This notebook computes User Popularity Deviation (UPD) for the **unadjusted ALS baseline** using the real prepared inputs from `popularity-profiles.ipynb` and the `top-10-baseline` table exported by `als-baseline.ipynb`.

The metric code in section 2 is copied **unchanged** from `upd-evaluation.ipynb` (its `validateUserMismatchScores` and `evaluateUPD` cells), so this notebook uses exactly the evaluator the team already validated on synthetic data. That notebook is not modified.

**Inputs** (the newest complete run of each is used unless overridden in section 3):

| Input | Source |
|---|---|
| `user-popularity-profiles`, `movie-popularity-groups`, `user-segments`, `user-history-buckets`, `evaluation-users` | `outputs/popularity-profiles/<run>/` |
| `top-10-baseline` | `outputs/als-baseline/<run>/` |

Both runs must use the same split (seed 42, per-user 80/20). Lower UPD means the recommendation lists match users' historical popularity mix more closely.

## 1. Environment

In [ ]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`

import org.apache.spark.sql.SparkSession

In [ ]:
val spark = SparkSession.builder()
  .appName("CS 7265 - Baseline UPD")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val checkCount = spark.range(3).count()

assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(checkCount == 3L)

println(s"Spark: ${spark.version}")
println(s"Scala: $scalaVersion")
println(s"Java: $javaVersion")
println(s"Environment check passed: $checkCount synthetic rows.")

## 2. UPD evaluator (copied unchanged from `upd-evaluation.ipynb`)

The next two cells are the validation helper and `evaluateUPD`, copied verbatim. They were tested there on synthetic inputs (identical mixes score 0, disjoint mixes score 1, equal segment weighting, and 27 input-rejection checks).

In [ ]:
import spark.implicits._
import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.functions.{avg, col, count, isnan, lit}

def validateUserMismatchScores(scores: DataFrame): Unit = {
  val invalidRows = scores.filter(
    col("userId").isNull ||
    col("segment").isNull ||
    col("mismatch").isNull ||
    isnan(col("mismatch")) ||
    col("mismatch") < 0.0 ||
    col("mismatch") > 1.0
  )
  assert(invalidRows.count() == 0L,
    "Every row must have a user, a segment, and a finite mismatch score between 0 and 1.")

  val duplicateUsers = scores.groupBy("userId").count()
    .filter(col("count") > 1L)
  assert(duplicateUsers.count() == 0L,
    "Each user must contribute exactly one mismatch score.")

  val expectedSegments = Set("blockbuster-focused", "diverse", "niche-focused")
  val actualSegments = scores.select("segment").distinct()
    .as[String].collect().toSet
  assert(actualSegments == expectedSegments,
    "Evaluation must contain exactly the three expected user segments.")
}

In [ ]:
import org.apache.spark.sql.Column
import org.apache.spark.sql.types.{DataType, IntegerType, DoubleType, StringType}
import org.apache.spark.sql.functions.{abs, greatest, least, log2, sum, when}

def requireColumns(
  table: DataFrame,
  label: String,
  fields: Seq[(String, DataType)]
): Unit = {
  fields.foreach { case (name, dataType) =>
    require(table.columns.count(_ == name) == 1,
      s"$label must contain exactly one column named $name.")
    require(table.schema(name).dataType == dataType,
      s"$label.$name must have type $dataType; convert it explicitly before evaluation.")
  }
}

def requireUniqueKeys(table: DataFrame, keys: Seq[String], label: String): Unit = {
  require(table.filter(keys.map(k => col(k).isNull).reduce(_ || _)).limit(1).count() == 0L,
    s"$label contains missing keys.")
  require(table.groupBy(keys.map(col): _*).count()
    .filter(col("count") > 1L).limit(1).count() == 0L,
    s"$label contains duplicate keys.")
}

def requireUserCoverage(table: DataFrame, expected: DataFrame, label: String): Unit = {
  val actual = table.select("userId").distinct()
  require(expected.join(actual, Seq("userId"), "left_anti").limit(1).count() == 0L,
    s"$label is missing expected users.")
  require(actual.join(expected, Seq("userId"), "left_anti").limit(1).count() == 0L,
    s"$label contains unexpected users; select the evaluation cohort explicitly.")
}

// Builds Spark expressions on the driver; no notebook closure is sent to workers.
def klContributionColumn(proportion: Column, reference: Column): Column = {
  when(proportion === 0.0, lit(0.0))
    .otherwise(proportion * log2(proportion / reference))
}

def evaluateUPD(
  profiles: DataFrame,
  movieGroups: DataFrame,
  recommendations: DataFrame,
  segments: DataFrame,
  expectedUsers: DataFrame,
  listSize: Int = 10
): (DataFrame, DataFrame, DataFrame) = {
  require(listSize > 0, "Recommendation list size must be positive.")
  val epsilon = 1e-9
  requireColumns(profiles, "profiles", Seq(
    "userId" -> IntegerType, "head" -> DoubleType,
    "mid" -> DoubleType, "tail" -> DoubleType))
  requireColumns(movieGroups, "movieGroups", Seq(
    "movieId" -> IntegerType, "popularityGroup" -> StringType))
  requireColumns(recommendations, "recommendations", Seq(
    "userId" -> IntegerType, "movieId" -> IntegerType))
  requireColumns(segments, "segments", Seq("userId" -> IntegerType, "segment" -> StringType))
  requireColumns(expectedUsers, "expectedUsers", Seq("userId" -> IntegerType))

  val cohort = expectedUsers.select("userId")
  val profileData = profiles.select("userId", "head", "mid", "tail")
  val groupData = movieGroups.select("movieId", "popularityGroup")
  val recommendationData = recommendations.select("userId", "movieId")
  val segmentData = segments.select("userId", "segment")

  requireUniqueKeys(cohort, Seq("userId"), "expectedUsers")
  require(cohort.limit(1).count() == 1L, "The evaluation cohort must not be empty.")
  requireUniqueKeys(profileData, Seq("userId"), "profiles")
  requireUniqueKeys(groupData, Seq("movieId"), "movieGroups")
  requireUniqueKeys(segmentData, Seq("userId"), "segments")
  requireUniqueKeys(recommendationData, Seq("userId", "movieId"), "recommendations")
  requireUserCoverage(profileData, cohort, "profiles")
  requireUserCoverage(segmentData, cohort, "segments")
  requireUserCoverage(recommendationData, cohort, "recommendations")

  val invalidProportion = Seq("head", "mid", "tail").map { name =>
    col(name).isNull || isnan(col(name)) || col(name) < 0.0 || col(name) > 1.0
  }.reduce(_ || _)
  require(profileData.filter(invalidProportion ||
    abs(col("head") + col("mid") + col("tail") - 1.0) > epsilon)
    .limit(1).count() == 0L, "Historical mixes must be finite, nonnegative, and sum to 1.")
  require(groupData.filter(col("popularityGroup").isNull ||
    !col("popularityGroup").isin("H", "M", "T")).limit(1).count() == 0L,
    "Movie popularity labels must be H, M, or T.")
  require(segmentData.filter(col("segment").isNull ||
    !col("segment").isin("blockbuster-focused", "diverse", "niche-focused"))
    .limit(1).count() == 0L, "User segment labels are missing or unknown.")
  require(segmentData.select("segment").distinct().count() == 3L,
    "All three user segments must be represented.")
  require(recommendationData.groupBy("userId").count()
    .filter(col("count") =!= listSize.toLong).limit(1).count() == 0L,
    s"Each user must have exactly $listSize distinct recommended movies.")
  require(recommendationData.select("movieId").distinct()
    .join(groupData.select("movieId"), Seq("movieId"), "left_anti")
    .limit(1).count() == 0L, "Some recommended movies have no popularity label.")

  val recommendationMixes = recommendationData.join(groupData, Seq("movieId"), "inner")
    .groupBy("userId")
    .agg(
      count("*").as("recommendationCount"),
      (sum(when(col("popularityGroup") === "H", 1.0).otherwise(0.0)) / count("*")).as("qHead"),
      (sum(when(col("popularityGroup") === "M", 1.0).otherwise(0.0)) / count("*")).as("qMid"),
      (sum(when(col("popularityGroup") === "T", 1.0).otherwise(0.0)) / count("*")).as("qTail")
    )
  val joined = profileData.join(recommendationMixes, Seq("userId"), "inner")
    .join(segmentData, Seq("userId"), "inner")
  val divergence = Seq("head" -> "qHead", "mid" -> "qMid", "tail" -> "qTail")
    .map { case (pName, qName) =>
      val p = col(pName)
      val q = col(qName)
      val m = (p + q) / 2.0
      (klContributionColumn(p, m) + klContributionColumn(q, m)) / 2.0
    }.reduce(_ + _)
  val rawScores = joined.withColumn("mismatch", divergence)
  require(rawScores.filter(col("mismatch").isNull || isnan(col("mismatch")) ||
    col("mismatch") < -epsilon || col("mismatch") > 1.0 + epsilon)
    .limit(1).count() == 0L, "Calculated mismatch is outside its numerical bounds.")
  // Clamp only rounding-sized excursions after rejecting larger violations.
  val userScores = rawScores.withColumn("mismatch",
    greatest(lit(0.0), least(lit(1.0), col("mismatch"))))
  requireUserCoverage(userScores, cohort, "computed scores")
  validateUserMismatchScores(userScores)
  val segmentScores = userScores.groupBy("segment")
    .agg(count("*").as("userCount"), avg("mismatch").as("upd"))
  val overallScore = segmentScores.agg(avg("upd").as("overallUPD"))
  (userScores, segmentScores, overallScore)
}

## 3. Locate and load the real inputs

By default the newest run folder that contains the required tables is used. To pin specific runs, set `popRunOverride` and `baselineRunOverride` to the run folder names (for example `Some("2026-10-28T02-14-20.965962900Z")`). The chosen paths are printed so they can be recorded in the report.

In [ ]:
import java.io.File

val popRunOverride: Option[String] = None
val baselineRunOverride: Option[String] = None

def findBase(relative: String): File =
  Seq(relative, s"../$relative").map(new File(_)).find(_.isDirectory)
    .getOrElse(throw new RuntimeException(s"Could not find $relative; run from the repository root or notebooks folder."))

def pickRun(base: File, pinnedName: Option[String], required: Seq[String]): File = {
  pinnedName match {
    case Some(name) =>
      val dir = new File(base, name)
      require(dir.isDirectory, s"Run folder not found: ${dir.getPath}")
      dir
    case None =>
      val complete = base.listFiles().filter(_.isDirectory)
        .filter(d => required.forall(r => new File(d, r).isDirectory))
        .sortBy(_.getName)
      require(complete.nonEmpty, s"No complete run found in ${base.getPath} (needs: ${required.mkString(", ")}).")
      complete.last
  }
}

val popRun = pickRun(findBase("outputs/popularity-profiles"), popRunOverride,
  Seq("user-popularity-profiles", "movie-popularity-groups", "user-segments", "user-history-buckets", "evaluation-users"))
val baselineRun = pickRun(findBase("outputs/als-baseline"), baselineRunOverride, Seq("top-10-baseline"))

println(s"Popularity inputs: ${popRun.getPath}")
println(s"Baseline lists:    ${baselineRun.getPath}")
println(new String(java.nio.file.Files.readAllBytes(new File(popRun, "split-config.txt").toPath), "UTF-8"))

In [ ]:
val profiles = spark.read.parquet(new File(popRun, "user-popularity-profiles").getPath)
val movieGroups = spark.read.parquet(new File(popRun, "movie-popularity-groups").getPath)
val segments = spark.read.parquet(new File(popRun, "user-segments").getPath)
val historyBuckets = spark.read.parquet(new File(popRun, "user-history-buckets").getPath)
val expectedUsers = spark.read.parquet(new File(popRun, "evaluation-users").getPath)
val recommendations = spark.read.parquet(new File(baselineRun, "top-10-baseline").getPath)

println(s"profiles: ${profiles.count()}, movie labels: ${movieGroups.count()}, segments: ${segments.count()}, " +
  s"buckets: ${historyBuckets.count()}, cohort: ${expectedUsers.count()}, baseline rows: ${recommendations.count()}")
assert(recommendations.count() == 60400L, "Expected 10 recommendations for each of 6,040 users.")

## 4. Compute baseline UPD

`evaluateUPD` checks schemas, keys, user coverage, label coverage, and list sizes before scoring, and rejects inputs rather than repairing them. Overall UPD is the equally weighted mean of the three segment means.

In [ ]:
val (userScores, segmentScores, overallScore) = evaluateUPD(
  profiles, movieGroups, recommendations, segments, expectedUsers)
userScores.cache()

println("UPD by user segment (baseline ALS):")
segmentScores.orderBy("segment").show(truncate = false)
println("Overall UPD (mean of the three segment means):")
overallScore.show(truncate = false)

## 5. Where the mismatch comes from

Average Head/Mid/Tail share of users' histories compared with their baseline recommendation lists, by segment. A large gap in the Head column (lists much more Head-heavy than histories) is the popularity bias that calibration is meant to correct.

In [ ]:
import org.apache.spark.sql.functions.{avg, round => sround, count => scount}

userScores.groupBy("segment").agg(
  scount("*").as("users"),
  avg("head").as("histHead"), avg("qHead").as("recHead"),
  avg("mid").as("histMid"),   avg("qMid").as("recMid"),
  avg("tail").as("histTail"), avg("qTail").as("recTail"),
  avg("mismatch").as("upd")
).orderBy("segment").show(truncate = false)

userScores.agg(
  avg("head").as("histHead"), avg("qHead").as("recHead"),
  avg("mid").as("histMid"),   avg("qMid").as("recMid"),
  avg("tail").as("histTail"), avg("qTail").as("recTail")
).show(truncate = false)

## 6. UPD by history length

The history buckets (short < 50, medium 50 to 200, long > 200 training ratings) are the grouping the history-aware extension will be judged on. Segment and history length overlap heavily (few blockbuster-focused users have long histories), so the cross-table shows user counts alongside mean UPD. Treat cells with very few users as unreliable.

In [ ]:
val scoresWithBucket = userScores.join(historyBuckets.select("userId", "historyBucket"), Seq("userId"), "inner")
assert(scoresWithBucket.count() == userScores.count(), "Every scored user needs a history bucket.")

println("Mean UPD by history bucket:")
scoresWithBucket.groupBy("historyBucket").agg(scount("*").as("users"), avg("mismatch").as("upd"))
  .orderBy(when(col("historyBucket") === "short", 1).when(col("historyBucket") === "medium", 2).otherwise(3))
  .show(truncate = false)

println("Mean UPD by segment and history bucket:")
scoresWithBucket.groupBy("segment").pivot("historyBucket", Seq("short", "medium", "long"))
  .agg(avg("mismatch")).orderBy("segment").show(truncate = false)

println("User counts by segment and history bucket:")
scoresWithBucket.groupBy("segment").pivot("historyBucket", Seq("short", "medium", "long"))
  .count().orderBy("segment").show(truncate = false)

## 7. Save the results

Saved under the ignored `outputs/baseline-upd/<run timestamp>/`, with the input run identities, so each number in the report can be traced to the exact files that produced it.

In [ ]:
val resultsRoot = new File(findBase("outputs"), "baseline-upd")
val resultsRunId = java.time.Instant.now().toString.replace(":", "-")
val resultsDir = new File(resultsRoot, resultsRunId)

userScores.coalesce(1).write.mode("errorifexists").parquet(new File(resultsDir, "user-upd-scores").getPath)
segmentScores.coalesce(1).write.mode("errorifexists").parquet(new File(resultsDir, "segment-upd").getPath)
overallScore.coalesce(1).write.mode("errorifexists").parquet(new File(resultsDir, "overall-upd").getPath)

val provenance = Seq(
  "method=als-baseline-unadjusted",
  s"popularityInputs=${popRun.getPath}",
  s"baselineLists=${baselineRun.getPath}"
).mkString("\n") + "\n"
java.nio.file.Files.write(new File(resultsDir, "provenance.txt").toPath, provenance.getBytes("UTF-8"))

println(s"Saved results to ${resultsDir.getPath}")

## 8. Stop the Spark session

In [ ]:
userScores.unpersist(blocking = true)
spark.stop()
println("Spark session stopped.")